In [4]:
import torch

torch.linspace(0, 10, 41)

tensor([ 0.0000,  0.2500,  0.5000,  0.7500,  1.0000,  1.2500,  1.5000,  1.7500,
         2.0000,  2.2500,  2.5000,  2.7500,  3.0000,  3.2500,  3.5000,  3.7500,
         4.0000,  4.2500,  4.5000,  4.7500,  5.0000,  5.2500,  5.5000,  5.7500,
         6.0000,  6.2500,  6.5000,  6.7500,  7.0000,  7.2500,  7.5000,  7.7500,
         8.0000,  8.2500,  8.5000,  8.7500,  9.0000,  9.2500,  9.5000,  9.7500,
        10.0000])

In [6]:
idl = torch.nn.Identity()

t = torch.tensor([1, 2, 3, 4])

with torch.no_grad():
  print(idl(t))

tensor([1, 2, 3, 4])


In [ ]:
def make_schedule(timesteps, target_device):
    edges = torch.linspace(0, timesteps, timesteps + 1, dtype=torch.float64)
    cumulative = torch.cos(((edges / timesteps + 0.008) / 1.008) * math.pi / 2).square()
    cumulative = cumulative / cumulative[0]
    betas = (1 - cumulative[1:] / cumulative[:-1]).clamp(1e-5, 0.999).float()
    alphas = 1 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)
    previous = torch.cat([torch.ones(1), alpha_bars[:-1]])
    return {
        "beta": betas.to(target_device),
        "alpha": alphas.to(target_device),
        "alpha_bar": alpha_bars.to(target_device),
        "alpha_bar_prev": previous.to(target_device),
        "posterior_variance": (betas * (1 - previous) / (1 - alpha_bars)).to(target_device),
    }


In [11]:
import math

make_schedule(5, torch.device("cuda"))

{'beta': tensor([0.1013, 0.2795, 0.4736, 0.7241, 0.9990], device='cuda:0'),
 'alpha': tensor([0.8987, 0.7205, 0.5264, 0.2759, 0.0010], device='cuda:0'),
 'alpha_bar': tensor([8.9871e-01, 6.4748e-01, 3.4081e-01, 9.4046e-02, 9.4044e-05],
        device='cuda:0'),
 'alpha_bar_prev': tensor([1.0000, 0.8987, 0.6475, 0.3408, 0.0940], device='cuda:0'),
 'posterior_variance': tensor([0.0000, 0.0803, 0.2533, 0.5268, 0.9051], device='cuda:0')}

In [15]:
timesteps = 5
edges = torch.linspace(0, timesteps, timesteps + 1, dtype=torch.float64)
cumulative = torch.cos(((edges / timesteps + 0.008) / 1.008) * math.pi / 2).square()

(edges / timesteps + 0.008), (edges / timesteps + 0.008) / 1.008, torch.cos(((edges / timesteps + 0.008) / 1.008) * math.pi / 2)

(tensor([0.0080, 0.2080, 0.4080, 0.6080, 0.8080, 1.0080], dtype=torch.float64),
 tensor([0.0079, 0.2063, 0.4048, 0.6032, 0.8016, 1.0000], dtype=torch.float64),
 tensor([9.9992e-01, 9.4793e-01, 8.0460e-01, 5.8374e-01, 3.0664e-01, 6.1232e-17],
        dtype=torch.float64))

In [ ]:
cumulative / cumulative[0]

tensor([1.0000e+00, 8.9871e-01, 6.4748e-01, 3.4081e-01, 9.4046e-02, 3.7500e-33],
       dtype=torch.float64)

In [18]:
1 - cumulative[1:], cumulative[:-1]

(tensor([0.1014, 0.3526, 0.6592, 0.9060, 1.0000], dtype=torch.float64),
 tensor([0.9998, 0.8986, 0.6474, 0.3408, 0.0940], dtype=torch.float64))